# 🎨 ConPL: Consistent Prototype Learning (ACL 2023 Official Reproduction)
### Few-Shot Continual Relation Extraction on FewRel 5-shot (8 tasks × 10 relations, 6 sequences)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/main/notebooks/conpl_fewrel_5shot.ipynb)

Notebook này tái lập kết quả của paper **Consistent Prototype Learning for Few-Shot Continual Relation Extraction (ACL 2023)** trên Google Colab với NVIDIA GPU:
- **Dataset:** FewRel 5-shot (8 tasks, mỗi task 10 relations, 5-shot toàn bộ 8 tasks).
- **Backbone:** `bert-base-uncased` với Prompt-based Tuning (`e1 mask e2 . `).
- **Protocol gốc:** 6 sequences khởi tạo từ `rel_cluster_label_0.npy` với seeds [100, 110, 120, 130, 140, 150].
- **Tiêu chí nghiệm thu (Table 1 Paper):** Cumulative Whole Accuracy tại $T_8 \approx 85.77\%$.


## 1. Kiểm tra GPU Preflight (NVIDIA CUDA Verification)


In [ ]:
import torch

print('=' * 60)
print('KIỂM TRA THIẾT BỊ TÍNH TOÁN (GPU PREFLIGHT)')
print('=' * 60)
if torch.cuda.is_available():
    print(f'✓ CUDA Khả dụng: True')
    print(f'✓ GPU Thiết bị: {torch.cuda.get_device_name(0)}')
    print(f'✓ Bộ nhớ VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    !nvidia-smi
else:
    print('⚠️ CẢNH BÁO: CUDA KHÔNG KHẢ DỤNG! Bạn đang chạy trên CPU.')
    print('👉 Hướng dẫn bật GPU: Chọn Runtime trên thanh menu -> Change runtime type -> Chọn T4 GPU -> Lưu.')


## 2. Thiết lập Codebase & Môi trường Dependencies


In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/vinhlq2512/research-code-lab.git"
REPO_DIR = Path("/content/research-code-lab")
REPO_BRANCH = "feat/reproduce-conpl-baseline"

def git(*args, check=True):
    return subprocess.run(
        ["git", "-C", str(REPO_DIR), *args],
        check=check,
        text=True,
        capture_output=True,
    )

if not REPO_DIR.exists():
    print(f"Đang clone nhánh {REPO_BRANCH} từ GitHub...")
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"{REPO_DIR} tồn tại nhưng không phải Git repository")
    print(f"Đang cập nhật nhánh {REPO_BRANCH}...")
    git("fetch", "--quiet", "origin", REPO_BRANCH)
    local_branch = git("show-ref", "--verify", "--quiet", f"refs/heads/{REPO_BRANCH}", check=False)
    if local_branch.returncode == 0:
        git("checkout", REPO_BRANCH)
    else:
        git("checkout", "--track", "-b", REPO_BRANCH, f"origin/{REPO_BRANCH}")
    git("pull", "--ff-only", "origin", REPO_BRANCH)

commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()
print(f"✓ Đang dùng {REPO_BRANCH} @ {commit}")

%cd /content/research-code-lab/paper-reproductions/ConPL

# Dùng Python 3.10 riêng để tương thích với dependencies legacy của ConPL.
!pip install -q uv
!uv python install 3.10
!uv venv --python 3.10 .venv
!uv pip install --python .venv/bin/python torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python .venv/bin/python "huggingface-hub>=0.8.1,<1.0" "tokenizers>=0.11.1,<0.14" "numpy<1.24" scikit-learn wordninja six tqdm


## 3. Kiểm tra Dữ liệu FewRel & Sampler Integrity


In [ ]:
# Kiểm tra sampler và tính toàn vẹn dữ liệu thông qua môi trường ảo Python 3.10 (.venv)
!.venv/bin/python -c "\
import sys\n\
import json\n\
import torch\n\
import numpy as np\n\
sys.path.insert(0, '.')\n\
from encode import BERTSentenceEncoderPrompt\n\
from dataprocess import data_sampler_bert_prompt_deal_first_task\n\
with open('config/config_fewrel_5and10.json', 'r') as f:\n\
    cfg = json.load(f)\n\
cfg['rel_cluster_label'] = 'data/fewrel/CFRLdata_10_100_10_5/rel_cluster_label_0.npy'\n\
cfg['training_file'] = 'data/fewrel/CFRLdata_10_100_10_5/train_0.txt'\n\
cfg['valid_file'] = 'data/fewrel/CFRLdata_10_100_10_5/valid_0.txt'\n\
cfg['test_file'] = 'data/fewrel/CFRLdata_10_100_10_5/test_0.txt'\n\
cfg['first_task_k-way'] = 10\n\
cfg['k-shot'] = 5\n\
cfg['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'\n\
encoder = BERTSentenceEncoderPrompt(cfg)\n\
sampler = data_sampler_bert_prompt_deal_first_task(cfg, encoder.tokenizer, 'e1 mask e2 . ')\n\
sampler.set_seed(100)\n\
print('=' * 60)\n\
print('XÁC THỰC MÔI TRƯỜNG & DỮ LIỆU CONPL THÀNH CÔNG')\n\
print('=' * 60)\n\
print(f'✓ Thiết bị tính toán: {cfg["device"]}')\n\
print(f'✓ Số lượng relation: {len(sampler.id2rel) - 1}')\n\
print('✓ Data Sampler khởi tạo hoàn toàn chính xác!')"


## 4. Smoke test rồi chạy Full Benchmark ConPL


In [ ]:
import os
from pathlib import Path

# smoke_test: 1 sequence x 2 tasks; full_benchmark: 6 sequences x 8 tasks.
MODE = "smoke_test"  # Đổi sang "full_benchmark" sau khi smoke test qua.
if MODE == "smoke_test":
    sequences, tasks = 1, 2
    output_dir = Path("../../results/conpl/colab_smoke")
else:
    sequences, tasks = 6, 8
    output_dir = Path("../../results/conpl/colab_run_6seq")

os.environ["CONPL_DEVICE"] = "cuda"
os.environ["CONPL_SEQUENCES"] = str(sequences)
os.environ["CONPL_TASKS"] = str(tasks)
os.environ["CONPL_OUTPUT_DIR"] = str(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
log_path = output_dir / "run.log"

print(f"🚀 Bắt đầu ConPL ({MODE}) trên GPU Colab...")
print(f"📁 Kết quả: {output_dir}")
print(f"📝 Log: {log_path}")

# bash pipefail makes the notebook cell fail if the Python process fails.
!bash -o pipefail -c '.venv/bin/python fewrel_5shot.py 2>&1 | tee "{log_path}"'


## 5. Tổng hợp Kết quả & Đối chiếu Paper Table 1


In [ ]:
import os
import json
import pandas as pd
from pathlib import Path

paper_reported = {
    'T1': 95.72, 'T2': 93.53, 'T3': 91.31, 'T4': 89.95,
    'T5': 88.93, 'T6': 88.39, 'T7': 87.43, 'T8': 85.77
}

print('=' * 60)
print('ĐỐI CHIẾU KẾT QUẢ VỚI PAPER CONPL (ACL 2023 TABLE 1)')
print('=' * 60)

output_dir = Path(os.environ.get('CONPL_OUTPUT_DIR', '../../results/conpl/colab_run_6seq'))
metrics_files = sorted(output_dir.glob('seed_*/task_metrics.jsonl'))

if metrics_files:
    seq_accs = {}
    for mf in metrics_files:
        seed_name = mf.parent.name
        with open(mf) as f:
            records = [json.loads(line) for line in f]
        seq_accs[seed_name] = {f"T{r['task']}": r['accuracy_whole'] * 100 for r in records}
    
    df = pd.DataFrame(seq_accs).T
    df.loc['Colab Mean'] = df.mean()
    df.loc['Paper Table 1'] = pd.Series(paper_reported)
    if 'Colab Mean' in df.index and 'Paper Table 1' in df.index:
        df.loc['Diff (Colab - Paper)'] = df.loc['Colab Mean'] - df.loc['Paper Table 1']
    display(df.round(2))
else:
    for k, v in paper_reported.items():
        print(f'{k}: Paper={v:.2f}%')
